# 1.3 Randomized Measurement with Error Mitigation

## Entanglement Entropy

```{seealso}

You can download the Jupyter Notebook for this section from the link {nb-download}`./pp_1_3_entropy_randomized_mitigated.ipynb`.
```

This method is based on [Probing Rényi entanglement entropy via randomized measurements](https://www.science.org/doi/abs/10.1126/science.aau4963) with deplorizing error mitigation by [Simple mitigation of global depolarizing errors in quantum simulations](https://link.aps.org/doi/10.1103/PhysRevE.104.035309).


## `randomized_entangled_entropy_mitigated`


Similar to the function `randomized_entangled_entropy` introduced in [1.2 Randomized Measurement](./2_randomized_measure.ipynb),
but not just calculating the purity and entropy of the target system,
this function also calculates the purity and entropy of the all system,
and then mitigates the purity and entropy of the target system by the purity of the all system with depolarizing error mitigation method.

For `randomized_entangled_entropy` return a single dictionary `TargetSystemResult` for target system,
but for `randomized_entangled_entropy_mitigated`,
it returns three dictionaries: `TargetSystemResult` for target system,
`AllSystemResult` for all system,
and `MitigatedResult` for mitigated result,
then package them together in a tuple.

This function requires only three arguments: `shots`, `counts`, and `selected_classical_registers`. All other arguments are optional.

Note that `selected_classical_registers` expects a list of <u>bitstring indices</u>, **NOT** <u>qubit indices</u>, as these are distinct.

The following is the arguments of the function:

```python
def randomized_entangled_entropy_mitigated(
    shots: int,
    counts: list[dict[str, int]],
    selected_classical_registers: Iterable[int] | None = None,
    existed_all_system: AllSystemResult | None = None,
    backend: PostProcessingBackendLabel = DEFAULT_PROCESS_BACKEND,
    pbar: tqdm.tqdm | None = None,
) -> tuple[TargetSystemResult, AllSystemResult, MitigatedResult]:
    """Calculate entangled entropy.

    Args:
        shots (int):
            Shots of the counts.
        counts (list[dict[str, int]]):
            Counts from randomized measurement results.
        selected_classical_registers (Iterable[int] | None, optional):
            The list of **the index of the selected_classical_registers**. Defaults to None.
        existed_all_system (AllSystemResult | None, optional):
            Existing all system source.
            If there is known all system result, then you can put it here
            to save a lot of time on calculating all system for no matter
            what partition you are using, their all system result is the same.
            This can save a lot of time
            Defaults to None.
        backend (PostProcessingBackendLabel, optional):
            Backend for the process. Defaults to DEFAULT_PROCESS_BACKEND.
        pbar (tqdm.tqdm | None, optional):
            The progress bar API,
            you can use put a `tqdm.tqdm <https://tqdm.github.io/>` object here.
            This function will update the progress bar description.
            Defaults to None.

    Returns:
        The target system result, all system result, and mitigated result.
    """
```

The target system, same as the one in `randomized_entangled_entropy`:

```python
class TargetSystemResult(TypedDict):
    """The return type of the post-processing for entangled entropy."""

    purity: FloatType
    """The purity of the system."""
    entropy: FloatType
    """The entropy of the system."""
    purity_sd: FloatType
    """The standard deviation of the purity."""
    entropy_sd: FloatType
    """The standard deviation of the entropy."""
    purity_cells: dict[int, np.float64] | dict[int, float]
    """The purity of each single count."""

    num_classical_registers: int
    """The number of classical registers."""
    classical_registers: list[int] | None
    """The list of the index of the selected classical registers."""
    classical_registers_actually: list[int]
    """The list of the index of the selected classical registers which is actually used."""

    taking_time: float
    """The calculation time."""
    counts_num: int
    """The number of counts."""
```

The all system, it data structure extends from `TargetSystemResult`:

```python
class AllSystemResult(TargetSystemResult):
    """The return type of the post-processing for entangled entropy."""

    preparing_datetime: str
    """The datetime string when preparing the all system result."""
    result_hash_id: str
    """The hash id of the result for verification."""
    all_system_source: Literal["independent"] | str
    """The name of source of all system.

    - `independent`: The all system is calculated independently.
    """
```

The mitigated result, it only contains the mitigated info:

```python
class MitigatedResult(TypedDict):
    """The return type of the post-processing for entangled entropy with error mitigation."""

    # mitigated info
    error_rate: AllowedMitigatedInput
    """The error rate of the measurement from depolarizing error migigation calculated."""
    mitigated_purity: AllowedMitigatedInput
    """The mitigated purity."""
    mitigated_entropy: AllowedMitigatedInput
    """The mitigated entropy."""
```


### Dummy Data


```{warning}

The dummy data is just some random bitstrings for the demonstration of data structure.
They have no physical meaning.
```

```{eval-rst}

.. seealso::

   You can download the dummy data for this section from the link :download:`./easy-dummy.json`.
```


In [ ]:
import json

with open("./easy-dummy.json", "r") as f:
    easy_dummy = json.load(f)
large_dummy_list = [easy_dummy["0"] for _ in range(100)]

### Simple Example


In [2]:
from qurry.process.randomized_measure import randomized_entangled_entropy_mitigated

target_system_1, all_system_1, mitigated_1 = randomized_entangled_entropy_mitigated(
    4096, large_dummy_list, list(range(6))
)

In [ ]:
from pprint import pprint

print("| result of 'randomized_entangled_entropy_mitigated' except for 'purity_cells'")
pprint({k: v for k, v in target_system_1.items() if k != "purity_cells"})

print("-" * 50)
print("| Length of 'purity_cells':", len(target_system_1["purity_cells"]))
print("| Some result of 'purity_cells'")
pprint(dict(list(target_system_1["purity_cells"].items())[:10]))


| result of 'randomized_entangled_entropy_mitigated' except for purity_cells
{'classical_registers': [0, 1, 2, 3, 4, 5],
 'classical_registers_actually': [0, 1, 2, 3, 4, 5],
 'counts_num': 100,
 'entropy': np.float64(-0.08786065308638322),
 'entropy_sd': np.float64(0.0),
 'num_classical_registers': 8,
 'purity': np.float64(1.0627930164337158),
 'purity_sd': np.float64(0.0),
 'taking_time': 0.001834928}
--------------------------------------------------
| Length of purity_cells: 100
| Some result of 'purity_cells'
{0: 1.0627930164337158,
 9: 1.0627930164337158,
 47: 1.0627930164337158,
 60: 1.0627930164337158,
 61: 1.0627930164337158,
 73: 1.0627930164337158,
 77: 1.0627930164337158,
 83: 1.0627930164337158,
 94: 1.0627930164337158,
 95: 1.0627930164337158}


In [ ]:
print("| result of 'randomized_entangled_entropy_mitigated' except for 'purity_cells'")
pprint(
    {
        k: v
        for k, v in all_system_1.items()
        if k != "purity_cells" and k in target_system_1
    }
)

print("-" * 50)
print(
    "| Extended information of 'randomized_entangled_entropy_mitigated' for all system"
)
pprint(
    {
        k: v
        for k, v in all_system_1.items()
        if k != "purity_cells" and k not in target_system_1
    }
)

print("-" * 50)
print("| Length of purity_cells:", len(all_system_1["purity_cells"]))
print("| Some result of 'purity_cells'")
pprint(dict(list(all_system_1["purity_cells"].items())[:10]))

| result of 'randomized_entangled_entropy_mitigated' except for purity_cells
{'classical_registers': None,
 'classical_registers_actually': [0, 1, 2, 3, 4, 5, 6, 7],
 'counts_num': 100,
 'entropy': np.float64(0.9461940705953849),
 'entropy_sd': np.float64(0.0),
 'num_classical_registers': 8,
 'purity': np.float64(0.5189998149871826),
 'purity_sd': np.float64(0.0),
 'taking_time': 0.007772002}
--------------------------------------------------
| Extended information of 'randomized_entangled_entropy_mitigated' for all system
{'all_system_source': 'independent',
 'preparing_datetime': '2026-04-08 02:17:30',
 'result_hash_id': '0ce1e038598b4ebc12f6191ab6843ea5a3704b1e'}
--------------------------------------------------
| Length of purity_cells: 100
| Some result of 'purity_cells'
{13: 0.5189998149871826,
 16: 0.5189998149871826,
 28: 0.5189998149871826,
 34: 0.5189998149871826,
 42: 0.5189998149871826,
 43: 0.5189998149871826,
 66: 0.5189998149871826,
 85: 0.5189998149871826,
 86: 0.51899

In [5]:
print("| result of mitigated randomized_entangled_entropy")
pprint(mitigated_1)

| result of mitigated randomized_entangled_entropy
{'error_rate': np.float64(0.2808939301105586),
 'mitigated_entropy': np.float64(-1.0290289944568636),
 'mitigated_purity': np.float64(2.0406503299038543)}


### With Existing All System Data


In [6]:
target_system_2, all_system_2, mitigated_2 = randomized_entangled_entropy_mitigated(
    4096,
    large_dummy_list,
    list(range(6)),
    existed_all_system=all_system_1,
)

In [7]:
print("| result of 'randomized_entangled_entropy_mitigated' except for purity_cells")
pprint({k: v for k, v in target_system_2.items() if k != "purity_cells"})

print("-" * 50)
print("| Length of purity_cells:", len(target_system_2["purity_cells"]))
print("| Some result of 'purity_cells'")
pprint(dict(list(target_system_2["purity_cells"].items())[:10]))

| result of 'randomized_entangled_entropy_mitigated' except for purity_cells
{'classical_registers': [0, 1, 2, 3, 4, 5],
 'classical_registers_actually': [0, 1, 2, 3, 4, 5],
 'counts_num': 100,
 'entropy': np.float64(-0.08786065308638322),
 'entropy_sd': np.float64(0.0),
 'num_classical_registers': 8,
 'purity': np.float64(1.0627930164337158),
 'purity_sd': np.float64(0.0),
 'taking_time': 0.001337558}
--------------------------------------------------
| Length of purity_cells: 100
| Some result of 'purity_cells'
{14: 1.0627930164337158,
 20: 1.0627930164337158,
 27: 1.0627930164337158,
 37: 1.0627930164337158,
 41: 1.0627930164337158,
 56: 1.0627930164337158,
 66: 1.0627930164337158,
 67: 1.0627930164337158,
 88: 1.0627930164337158,
 94: 1.0627930164337158}


In [8]:
print("| result of 'randomized_entangled_entropy_mitigated' except for purity_cells")
pprint(
    {
        k: v
        for k, v in all_system_2.items()
        if k != "purity_cells" and k in target_system_1
    }
)

print("-" * 50)
print(
    "| Extended information of 'randomized_entangled_entropy_mitigated' for all system"
)
pprint(
    {
        k: v
        for k, v in all_system_2.items()
        if k != "purity_cells" and k not in target_system_1
    }
)

print(
    "| It should be the same as the result of 'randomized_entangled_entropy_mitigated' for all system in the first run"
)
pprint(
    {
        k: v
        for k, v in all_system_1.items()
        if k != "purity_cells" and k not in target_system_1
    }
)


print("-" * 50)
print("| Length of purity_cells:", len(all_system_2["purity_cells"]))
print("| Some result of 'purity_cells'")
pprint(dict(list(all_system_2["purity_cells"].items())[:10]))

| result of 'randomized_entangled_entropy_mitigated' except for purity_cells
{'classical_registers': None,
 'classical_registers_actually': [0, 1, 2, 3, 4, 5, 6, 7],
 'counts_num': 100,
 'entropy': np.float64(0.9461940705953849),
 'entropy_sd': np.float64(0.0),
 'num_classical_registers': 8,
 'purity': np.float64(0.5189998149871826),
 'purity_sd': np.float64(0.0),
 'taking_time': 0.007772002}
--------------------------------------------------
| Extended information of 'randomized_entangled_entropy_mitigated' for all system
{'all_system_source': 'AllSystemResult(preparing_datetime=2026-04-08 02:17:30, '
                      'result_hash_id=0ce1e038598b4ebc12f6191ab6843ea5a3704b1e)',
 'preparing_datetime': '2026-04-08 02:17:30',
 'result_hash_id': '0ce1e038598b4ebc12f6191ab6843ea5a3704b1e'}
| It should be the same as the result of 'randomized_entangled_entropy_mitigated' for all system in the first run
{'all_system_source': 'AllSystemResult(preparing_datetime=2026-04-08 02:17:30, '
    

In [9]:
print("| result of mitigated randomized_entangled_entropy")
pprint(mitigated_2)

| result of mitigated randomized_entangled_entropy
{'error_rate': np.float64(0.2808939301105586),
 'mitigated_entropy': np.float64(-1.0290289944568636),
 'mitigated_purity': np.float64(2.0406503299038543)}


### Integration wit your own progress bar


In [10]:
from tqdm import tqdm

all_counts_progress_01 = tqdm(
    [
        (4096, large_dummy_list, list(range(6))),
        (4096, large_dummy_list, list(range(2, 8))),
        (4096, large_dummy_list, list(range(7))),
        (4096, large_dummy_list, list(range(0, 7))),
        (4096, large_dummy_list, [0, 1, 2, 6, 7]),
        (4096, large_dummy_list, [3, 4, 5, 6, 7, 0, 1]),
    ],
    bar_format="| {desc} - {elapsed} < {remaining}",
)

test_result_2 = []
for tmp_shot, tmp_counts, tmp_partition in all_counts_progress_01:
    print(tmp_partition)
    test_result_2.append(
        randomized_entangled_entropy_mitigated(
            tmp_shot,
            tmp_counts,
            tmp_partition,
            existed_all_system=(None if len(test_result_2) == 0 else all_system_1),
            pbar=all_counts_progress_01,
        )
    )
    print(f"| partition: {tmp_partition}")
    print("| - taking_time:", test_result_2[-1][0]["taking_time"])
    print("| - taking_time_all_sys:", test_result_2[-1][1]["taking_time"])

| Using existing all system from 'AllSystemResult(preparing_datetime=2026-04-08 02:17:30, result_hash_id=0ce1e038598b4ebc12f6191ab6843ea5a3704b1e)' - 00:00 < 00:00

[0, 1, 2, 3, 4, 5]
| partition: [0, 1, 2, 3, 4, 5]
| - taking_time: 0.00146706
| - taking_time_all_sys: 0.00872752
[2, 3, 4, 5, 6, 7]
| partition: [2, 3, 4, 5, 6, 7]
| - taking_time: 0.002465906
| - taking_time_all_sys: 0.007772002
[0, 1, 2, 3, 4, 5, 6]
| partition: [0, 1, 2, 3, 4, 5, 6]
| - taking_time: 0.003793132
| - taking_time_all_sys: 0.007772002
[0, 1, 2, 3, 4, 5, 6]
| partition: [0, 1, 2, 3, 4, 5, 6]
| - taking_time: 0.003547262
| - taking_time_all_sys: 0.007772002
[0, 1, 2, 6, 7]
| partition: [0, 1, 2, 6, 7]
| - taking_time: 0.000809418
| - taking_time_all_sys: 0.007772002
[3, 4, 5, 6, 7, 0, 1]
| partition: [3, 4, 5, 6, 7, 0, 1]
| - taking_time: 0.003460823
| - taking_time_all_sys: 0.007772002


### Using Python backend

It will be slow. Yoy better think twice before using it.


In [11]:
all_counts_progress_02 = tqdm(
    [
        (4096, large_dummy_list, list(range(6))),
        (4096, large_dummy_list, list(range(2, 8))),
        (4096, large_dummy_list, list(range(7))),
        (4096, large_dummy_list, list(range(0, 7))),
        (4096, large_dummy_list, [0, 1, 2, 6, 7]),
        (4096, large_dummy_list, [3, 4, 5, 6, 7, 0, 1]),
    ],
    bar_format="| {desc} - {elapsed} < {remaining}",
)


test_result_3 = []
for tmp_shot, tmp_counts, tmp_partition in all_counts_progress_02:
    print(tmp_partition)
    test_result_3.append(
        randomized_entangled_entropy_mitigated(
            tmp_shot,
            tmp_counts,
            tmp_partition,
            existed_all_system=(None if len(test_result_2) == 0 else all_system_1),
            pbar=all_counts_progress_02,
            backend="Python",
        )
    )

| Calculate selected classical registers: [0, 1, 2, 3, 4, 5]. - 00:00 < ?

[0, 1, 2, 3, 4, 5]


| Calculate selected classical registers: [2, 3, 4, 5, 6, 7]. - 00:02 < 00:10                                                                                      

[2, 3, 4, 5, 6, 7]


| Calculate selected classical registers: [0, 1, 2, 3, 4, 5, 6]. - 00:04 < 00:08                                                                                   

...

| Calculate selected classical registers: [0, 1, 2, 6, 7]. - 00:09 < 00:04                                                                                         

[0, 1, 2, 6, 7]


| Calculate selected classical registers: [3, 4, 5, 6, 7, 0, 1]. - 00:11 < 00:02                                                                                   

[3, 4, 5, 6, 7, 0, 1]


| Using existing all system from 'AllSystemResult(preparing_datetime=2026-04-08 02:17:30, result_hash_id=0ce1e038598b4ebc12f6191ab6843ea5a3704b1e)' - 00:13 < 00:00


## Post-Process Availablities and Version Info


In [12]:
from qurry.process.status import AVAIBILITY_STATESHEET

AVAIBILITY_STATESHEET

 | Qurrium version: 1.0.0.dev2
---------------------------------------------------------------------------
 ### Qurrium Post-Processing
   - Backend Availability ................... Python Cython Rust   JAX   
 - randomized_measure
   - entangled_entropy.entropy_core_2 ....... Yes    No     Yes    No    
   - entangled_entropy_v1.entropy_core ...... Yes    Depr.  Yes    No    
   - wavefunction_overlap.echo_core_2 ....... Yes    No     Yes    No    
   - wavefunction_overlap_v1.echo_core ...... Yes    Depr.  Yes    No    
 - hadamard_test
   - purity_echo_core ....................... Yes    No     Yes    No    
 - magnet_square
   - magnsq_core ............................ Yes    No     Yes    No    
 - string_operator
   - strop_core ............................. Yes    No     Yes    No    
 - classical_shadow
   - rho_process ............................ Yes    No     Yes    No    
   - matrix_calculation ..................... Yes    No     No     Yes   
 - utils
   - randomized ....